# 08 — Baseline (Hyperparameter Default) di atas 182 Fitur (DDS) + Data Gabungan

**Tujuan:** Bangun titik pembanding "sebelum tuning" untuk pipeline 182-fitur
(164 kanonik + 18 DDS) di atas dataset gabungan terbesar saat ini (606 file
LIN + 1.390 file PBN = 49.755 board, lihat
[07_angelfire_expansion.ipynb](07_angelfire_expansion.ipynb)).

Semua model lain di eksperimen 2026-07-15 (XGBoost acc-tuned, LightGBM
class_weight) sudah memakai hyperparameter yang di-tuning. Notebook ini
sengaja TIDAK melakukan tuning apa pun — RF/XGBoost/LightGBM dipakai
dengan hyperparameter default `configs/config.yaml` murni (`RFModel()`,
`XGBModel()`, `LGBMModel()` tanpa argumen tambahan), supaya kontribusi
data tambahan + fitur DDS bisa dipisahkan secara bersih dari kontribusi
tuning hyperparameter/class_weight.

Notebook ini berjalan mandiri dari tahap ekstraksi data (parsing LIN+PBN)
sampai evaluasi test set — tidak bergantung pada dataset yang sudah
dibangun sebelumnya, meski cache DDS yang sudah dihitung tetap dipakai
ulang (tidak menghitung ulang DDS dari nol, itu memakan waktu ~2.9 jam).

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.parser import LINParser, PBNParser
from src.preprocessing.dataset_builder import build_dataset, load_splits
from src.models import RFModel, XGBModel, LGBMModel
from src.evaluation import evaluate, print_summary, compare_models

OUT_DIR = PROJECT_ROOT / "experiments" / "2026-07-15" / "outputs" / "baseline_dds_default"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Ekstraksi data (LIN + PBN)

In [2]:
raw_dir = PROJECT_ROOT / "data" / "raw"
raw_pbn_dir = PROJECT_ROOT / "data" / "raw_pbn"

lin_boards = LINParser().parse_directory(raw_dir)
pbn_boards = PBNParser().parse_directory(raw_pbn_dir)

print(f"Board dari LIN (BBO)    : {len(lin_boards):>6,}")
print(f"Board dari PBN (non-BBO): {len(pbn_boards):>6,}")
print(f"Total board mentah      : {len(lin_boards) + len(pbn_boards):>6,}")

Board dari LIN (BBO)    : 13,708
Board dari PBN (non-BBO): 47,712
Total board mentah      : 61,420


## 2. Bangun dataset 182-fitur (164 kanonik + 18 DDS)

Cache DDS (`data/combined_dds_cache_v3.csv`) sudah mencakup seluruh 49.755
board ini (dihitung di notebook 07) — dipakai ulang di sini, tidak dihitung
dari nol.

In [3]:
t0 = time.time()
build_dataset(
    raw_dir=raw_dir,
    output_dir=PROJECT_ROOT / "data" / "processed_combined",
    include_dds=True,
    dds_cache_path=PROJECT_ROOT / "data" / "combined_dds_cache_v3.csv",
    extra_boards=pbn_boards,
)
print(f"\nDataset dibangun dalam {time.time()-t0:.0f}s")

[1/5] Parsing LIN files...


      Boards loaded: 13708
      + 47712 extra boards from supplementary source(s)
[2/5] Extracting features...


      Feature rows : 61409


[3/5] Cleaning...


      Removed 11603 duplicate rows.
      Final rows   : 49806
      No missing values in features.
[4/5] Encoding labels...
      Classes      : 36
      Classes list : ['1C', '1D', '1H', '1N', '1S', '2C', '2D', '2H', '2N', '2S', '3C', '3D', '3H', '3N', '3S', '4C', '4D', '4H', '4N', '4S', '5C', '5D', '5H', '5N', '5S', '6C', '6D', '6H', '6N', '6S', '7C', '7D', '7H', '7N', '7S', 'PASS']
      Encoder saved: D:\Bridge-Prediction\data\processed_combined\label_encoder.pkl
[DDS] Menambahkan fitur Double-Dummy Solver...
      Loading cached DDS features: D:\Bridge-Prediction\data\combined_dds_cache_v3.csv


...

      Saved: D:\Bridge-Prediction\data\processed_combined\train.csv


      Saved: D:\Bridge-Prediction\data\processed_combined\val.csv


      Saved: D:\Bridge-Prediction\data\processed_combined\test.csv


      Saved: D:\Bridge-Prediction\data\processed_combined\full.csv
      Feature cols saved: D:\Bridge-Prediction\data\processed_combined\feature_columns.json



Dataset dibangun dalam 117s


In [4]:
df_train, df_val, df_test, feature_cols, le = load_splits(PROJECT_ROOT / "data" / "processed_combined")
X_train, y_train = df_train[feature_cols].values, df_train["label"].values
X_test, y_test = df_test[feature_cols].values, df_test["label"].values

print(f"Train: {X_train.shape}, Val: {df_val.shape}, Test: {X_test.shape}")
print(f"Jumlah kelas: {len(le.classes_)}")
print(f"Jumlah fitur: {len(feature_cols)} (164 kanonik + 18 DDS)")

D:\Bridge-Prediction\src\preprocessing\dataset_builder.py:296: DtypeWarning: Columns (0: _board_number) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(processed_dir / "train.csv")


Train: (34832, 182), Val: (7462, 192), Test: (7461, 182)
Jumlah kelas: 36
Jumlah fitur: 182 (164 kanonik + 18 DDS)


## 3. Latih model BASELINE — hyperparameter default, tanpa tuning

`RFModel()`, `XGBModel()`, `LGBMModel()` dipanggil TANPA argumen tambahan —
murni hyperparameter default `configs/config.yaml`. `RFModel` sudah
memakai `class_weight="balanced"` secara default sejak awal proyek
(bukan hasil tuning eksperimen 2026-07-15), XGBoost/LightGBM di sini TIDAK
memakai acc-tuning atau `class_weight` — itu baru ditambahkan di
notebook-notebook eksperimen berikutnya.

In [5]:
models = [RFModel(), XGBModel(), LGBMModel()]

print(f'{"Model":<15} {"Hyperparameters (default, config.yaml)"}')
print("-" * 70)
for m in models:
    print(f"{m.name:<15} {m.params}")

Model           Hyperparameters (default, config.yaml)
----------------------------------------------------------------------
RandomForest    {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 5, 'random_state': 42, 'n_jobs': -1, 'class_weight': 'balanced'}
XGBoost         {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42, 'n_jobs': -1, 'eval_metric': 'mlogloss', 'verbosity': 0}
LightGBM        {'n_estimators': 300, 'max_depth': -1, 'learning_rate': 0.05, 'num_leaves': 63, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}


In [6]:
results = []
for model in models:
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    res = evaluate(y_test, y_pred, y_proba, le, model_name=model.name)
    res["fit_seconds"] = round(elapsed, 1)
    results.append(res)
    print_summary(res)
    print(f"  (fit time: {elapsed:.1f}s)\n")


  RandomForest
  Accuracy          : 0.4684
  Precision (macro) : 0.3129
  Recall (macro)    : 0.3615
  F1 (macro)        : 0.3251
  F1 (weighted)     : 0.4854
  Top-3 Accuracy    : 0.7715
  Top-5 Accuracy    : 0.8737
  (fit time: 12.8s)




  XGBoost
  Accuracy          : 0.5745
  Precision (macro) : 0.4663
  Recall (macro)    : 0.3605
  F1 (macro)        : 0.3901
  F1 (weighted)     : 0.5540
  Top-3 Accuracy    : 0.8263
  Top-5 Accuracy    : 0.9047
  (fit time: 223.4s)



D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


D:\Bridge-Prediction\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



  LightGBM
  Accuracy          : 0.5633
  Precision (macro) : 0.4502
  Recall (macro)    : 0.3336
  F1 (macro)        : 0.3636
  F1 (weighted)     : 0.5392
  Top-3 Accuracy    : 0.8134
  Top-5 Accuracy    : 0.8869
  (fit time: 181.1s)



## 4. Bandingkan baseline vs kandidat ter-tuning (LightGBM+class_weight, XGBoost acc-tuned)

In [7]:
comparison = compare_models(results)
comparison.to_csv(OUT_DIR / "test_comparison.csv")

# Angka referensi dari 07_angelfire_expansion.ipynb (49.755 board, SUDAH di-tuning)
reference = {
    "XGBoost acc-tuned (REF, tuned)": {"accuracy": 0.5611, "f1_macro": 0.3416, "f1_weighted": 0.5322},
    "LightGBM class_weight (REF, tuned)": {"accuracy": 0.5639, "f1_macro": 0.4101, "f1_weighted": 0.5567},
}
ref_df = pd.DataFrame(reference).T
print("=== Kandidat ter-tuning (referensi, notebook 07) ===")
print(ref_df)
print()
print("=== Baseline default (notebook ini) ===")
print(comparison[["accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy"]])

=== Kandidat ter-tuning (referensi, notebook 07) ===
                                    accuracy  f1_macro  f1_weighted
XGBoost acc-tuned (REF, tuned)        0.5611    0.3416       0.5322
LightGBM class_weight (REF, tuned)    0.5639    0.4101       0.5567

=== Baseline default (notebook ini) ===
              accuracy  f1_macro  f1_weighted  top_3_accuracy  top_5_accuracy
model                                                                        
RandomForest  0.468436  0.325073     0.485363        0.771478        0.873743
XGBoost       0.574454  0.390107     0.554002        0.826297        0.904704
LightGBM      0.563329  0.363552     0.539209        0.813430        0.886878


In [8]:
import json

summary = {
    "generated": pd.Timestamp.now().isoformat(),
    "purpose": "untuned baseline (config.yaml defaults) on the 49,755-board / 182-feature combined dataset",
    "dataset_size": {
        "total_boards": len(df_train) + len(df_val) + len(df_test),
        "train": len(df_train), "val": len(df_val), "test": len(df_test),
        "n_features": len(feature_cols),
        "n_classes": len(le.classes_),
    },
    "results": {r["model"]: {k: r[k] for k in ("accuracy", "f1_macro", "f1_weighted", "top_3_accuracy", "top_5_accuracy", "fit_seconds")} for r in results},
    "tuned_reference": reference,
}
with open(OUT_DIR / "nb08_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "generated": "2026-07-17T01:03:26.575439",
  "purpose": "untuned baseline (config.yaml defaults) on the 49,755-board / 182-feature combined dataset",
  "dataset_size": {
    "total_boards": 49755,
    "train": 34832,
    "val": 7462,
    "test": 7461,
    "n_features": 182,
    "n_classes": 36
  },
  "results": {
    "RandomForest": {
      "accuracy": 0.4684358665058303,
      "f1_macro": 0.32507304544199334,
      "f1_weighted": 0.4853627426383334,
      "top_3_accuracy": 0.7714783541080285,
      "top_5_accuracy": 0.8737434660233213,
      "fit_seconds": 12.8
    },
    "XGBoost": {
      "accuracy": 0.5744538265648036,
      "f1_macro": 0.39010697269128886,
      "f1_weighted": 0.5540023622833717,
      "top_3_accuracy": 0.8262967430639324,
      "top_5_accuracy": 0.9047044632086851,
      "fit_seconds": 223.4
    },
    "LightGBM": {
      "accuracy": 0.5633293124246079,
      "f1_macro": 0.36355211906900753,
      "f1_weighted": 0.5392090632734822,
      "top_3_accuracy": 0.8

## 5. Kesimpulan

Notebook ini mengisolasi kontribusi **data (49.755 board) + fitur DDS**
murni, terpisah dari kontribusi **tuning hyperparameter/class_weight**
yang dieksplorasi di notebook-notebook lain. Hasil di atas dibandingkan
terhadap kandidat ter-tuning (XGBoost acc-tuned, LightGBM class_weight)
menunjukkan berapa banyak dari performa 56%+ accuracy / 0.41 F1 macro
berasal dari data+DDS saja vs berapa banyak dari tuning tambahan.